# Leaf Card Agent OCR Playground

这个 notebook 用来测试：`OCR / 摘录文本 -> rough json -> normalized json -> draft markdown -> 可选 cards`。

默认只写到 `scratch/leaf_agent_playground/`，不碰正式知识库产物。

为了减少 `429 RateLimit`，这里默认：
- 只测 2 个 `node_id`
- `max_tokens=900`
- 开启 429 自动重试
- 失败时直接打印模型原始输出

In [2]:
%pip install -U azure-ai-projects azure-identity openai nest_asyncio pydantic


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/anaconda3/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from leaf_card_agent import (
    DraftPayloadParseError,
    FoundryLeafCardDraftAgent,
    leaf_card_environment,
    summarize_result,
)

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'leaf_agent_playground'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps(leaf_card_environment(), ensure_ascii=False, indent=2))

{
  "project_endpoint": "https://teachagent-xumuchi-20260614.services.ai.azure.com/api/projects/proj-default",
  "model_deployment": "gpt-4o-mini",
  "structured_outputs_supported": "True"
}


## 1. 准备一段 OCR / 笔记文本

In [4]:
raw_text = '''
极值点偏移问题的七种解法
极值点偏移是导数题目中常见的一大类问题，尤其在选择填空和导数大题中经常出现。解决这类问题的关键在于简化已知函数，挖掘隐含条件。以下是七种常用的解法：
构造对数不等式
如果函数有两个零点，且在某点处导数小于零，则该点处函数值小于零。通过构造对数不等式，可以比较几何中的函数值与零点的大小关系。
三阶导法与构造对称
对于某些题目，构造对数不等式可能无法直接提取因式，此时可以考虑使用三阶导法。通过构造对称函数，可以简化计算过程。
对数均值不等式
对数均值不等式是一种常用的方法，尤其在解决极值点偏移问题时非常有效。通过比较几何中的函数值与零点的大小关系，可以得出结论。
比差值代换法
对于某些题目，直接使用对数不等式可能无法解决问题，此时可以考虑使用比差值代换法。通过构造函数并比较其大小关系，可以得出结论。
单调性法
如果函数在某点处单调递减，且该点处函数值大于零，则该点处导数小于零。通过单调性法，可以比较几何中的函数值与零点的大小关系。
中点切线斜率法
对于某些题目，通过构造中点切线斜率法可以简化计算过程。通过比较函数值与零点的大小关系，可以得出结论。
双变量法
对于某些题目，双变量法是一种有效的解决方法。通过构造双变量函数，可以简化计算过程，并得出正确的结论。
'''.strip()

print(raw_text[:400])

极值点偏移问题的七种解法
极值点偏移是导数题目中常见的一大类问题，尤其在选择填空和导数大题中经常出现。解决这类问题的关键在于简化已知函数，挖掘隐含条件。以下是七种常用的解法：
构造对数不等式
如果函数有两个零点，且在某点处导数小于零，则该点处函数值小于零。通过构造对数不等式，可以比较几何中的函数值与零点的大小关系。
三阶导法与构造对称
对于某些题目，构造对数不等式可能无法直接提取因式，此时可以考虑使用三阶导法。通过构造对称函数，可以简化计算过程。
对数均值不等式
对数均值不等式是一种常用的方法，尤其在解决极值点偏移问题时非常有效。通过比较几何中的函数值与零点的大小关系，可以得出结论。
比差值代换法
对于某些题目，直接使用对数不等式可能无法解决问题，此时可以考虑使用比差值代换法。通过构造函数并比较其大小关系，可以得出结论。
单调性法
如果函数在某点处单调递减，且该点处函数值大于零，则该点处导


## 2. 指定测试范围和轻量参数

In [5]:
node_ids = [
    'math.集合_映射_函数与导数.导数.利用导数判断单调性',
    'math.集合_映射_函数与导数.导数.利用导数求函数极值',
]

MODEL_DEPLOYMENT = None

agent_kwargs = {
    'max_tokens': 900,
    'temperature': 0.0,
    'max_retries': 3,
    'retry_backoff_seconds': 8.0,
}
if MODEL_DEPLOYMENT:
    agent_kwargs['model_deployment'] = MODEL_DEPLOYMENT

agent = FoundryLeafCardDraftAgent(**agent_kwargs)
print({'model': agent.model_deployment, 'max_tokens': agent.max_tokens})

{'model': 'gpt-4o-mini', 'max_tokens': 900}


## 3. 调用 agent

如果解析失败，这个单元会直接把模型原始输出打出来。

In [6]:
try:
    result = agent.generate(
        node_ids=node_ids,
        reference_text=raw_text,
        source='ocr_playground',
        max_tokens=900,
        extra_rule='这是 OCR 或网络笔记摘录场景。请尽量补全 document_title、document_summary、global_keywords、source_quality_notes、unresolved_points，并为每张卡补 source_excerpt、fit_reason、confidence_note。若原文表达模糊，请在 confidence_note 或 unresolved_points 中明确写出。严格只输出一个 JSON 对象，顶层必须包含 cards 字段。',
    )
    summary = summarize_result(result)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
except DraftPayloadParseError as exc:
    print(str(exc))
    print('\n===== RAW MODEL OUTPUT =====\n')
    print(exc.raw_output)
    result = None

{
  "response_source": "per_node_fallback",
  "document_title": null,
  "document_summary": null,
  "global_keywords": [],
  "source_quality_notes": [
    "batch_parse_failed_then_per_node_fallback_succeeded"
  ],
  "unresolved_points": [],
  "card_count": 2,
  "node_ids": [
    "math.集合_映射_函数与导数.导数.利用导数判断单调性",
    "math.集合_映射_函数与导数.导数.利用导数求函数极值"
  ]
}


## 4. 查看 rough json

In [7]:
if result is not None:
    print(json.dumps(result.rough_payload, ensure_ascii=False, indent=2))

{
  "document_title": null,
  "document_summary": null,
  "global_keywords": [],
  "source_quality_notes": [
    "batch_parse_failed_then_per_node_fallback_succeeded"
  ],
  "unresolved_points": [],
  "cards": [
    {
      "node_id": "math.集合_映射_函数与导数.导数.利用导数判断单调性",
      "card_type": "method_card",
      "title": null,
      "aliases": [],
      "keywords": [
        "导数",
        "单调性",
        "增函数",
        "减函数",
        "符号判别",
        "区间"
      ],
      "tags": [],
      "example_refs": [],
      "definition": null,
      "recognition_signals": [],
      "common_errors": [],
      "review_cue": "若已知 f'(x)>0 在区间 (a,b) 内成立，函数 f(x) 在该区间的单调性是什么？",
      "boundary": null,
      "core_idea": null,
      "formula": null,
      "applicable_conditions": [],
      "special_cases": [],
      "variable_notes": null,
      "derivation_hint": null,
      "method_goal": "判断函数在指定区间是单调递增还是单调递减",
      "trigger_signals": [
        "f'(x)>0",
        "f'(x)<0",
        "f'(x)=0",
        "导数符号变化

## 5. 查看规范化 entries

In [8]:
if result is not None:
    print(json.dumps(result.entries, ensure_ascii=False, indent=2))

[
  {
    "node_id": "math.集合_映射_函数与导数.导数.利用导数判断单调性",
    "method_goal": "判断函数在指定区间是单调递增还是单调递减",
    "review_cue": "若已知 f'(x)>0 在区间 (a,b) 内成立，函数 f(x) 在该区间的单调性是什么？",
    "card_type": "method_card",
    "trigger_signals": [
      "f'(x)>0",
      "f'(x)<0",
      "f'(x)=0",
      "导数符号变化点"
    ],
    "failure_modes": [
      "忽略导数不存在的点导致判断错误",
      "仅在单点检查导数符号而未划分区间",
      "把 f'(x)=0 的点误认为函数在该点不单调",
      "未考虑高阶导数导致误判平稳点"
    ],
    "keywords": [
      "导数",
      "单调性",
      "增函数",
      "减函数",
      "符号判别",
      "区间"
    ],
    "steps": [
      "计算函数的导数 f'(x)",
      "确定感兴趣的区间或全体定义域",
      "求解 f'(x)=0 或不存在的点，得到临界点",
      "将区间划分为若干子区间，选取代表点代入 f'(x) 判断符号",
      "若在子区间内 f'(x)>0，则函数在该子区间单调递增；若 f'(x)<0，则单调递减",
      "若在某子区间内 f'(x) 恒为零，则函数在该区间为常数或需进一步检查高阶导数"
    ],
    "title": "利用导数判断单调性"
  },
  {
    "node_id": "math.集合_映射_函数与导数.导数.利用导数求函数极值",
    "method_goal": "利用导数判别函数的极大值或极小值位置并求出极值大小",
    "review_cue": "已知函数 f(x)=..., 如何利用导数求其极大值或极小值？",
    "card_type": "method_card",
    "tri

## 6. 查看 draft markdown

In [9]:
if result is not None:
    print(result.draft_markdown)

### math.集合_映射_函数与导数.导数.利用导数判断单调性
- card_type: method_card
- title: 利用导数判断单调性
- keywords: 导数 | 单调性 | 增函数 | 减函数 | 符号判别 | 区间
- method_goal: 判断函数在指定区间是单调递增还是单调递减
- trigger_signals: f'(x)>0 | f'(x)<0 | f'(x)=0 | 导数符号变化点
- steps: 计算函数的导数 f'(x) | 确定感兴趣的区间或全体定义域 | 求解 f'(x)=0 或不存在的点，得到临界点 | 将区间划分为若干子区间，选取代表点代入 f'(x) 判断符号 | 若在子区间内 f'(x)>0，则函数在该子区间单调递增；若 f'(x)<0，则单调递减 | 若在某子区间内 f'(x) 恒为零，则函数在该区间为常数或需进一步检查高阶导数
- failure_modes: 忽略导数不存在的点导致判断错误 | 仅在单点检查导数符号而未划分区间 | 把 f'(x)=0 的点误认为函数在该点不单调 | 未考虑高阶导数导致误判平稳点
- review_cue: 若已知 f'(x)>0 在区间 (a,b) 内成立，函数 f(x) 在该区间的单调性是什么？

### math.集合_映射_函数与导数.导数.利用导数求函数极值
- card_type: method_card
- title: 利用导数求函数极值
- keywords: 导数 | 极值 | 临界点 | 一阶导数 | 二阶导数 | 单调性 | 凹凸性
- method_goal: 利用导数判别函数的极大值或极小值位置并求出极值大小
- trigger_signals: 需要求函数的最大值或最小值 | 已知函数表达式并可求导 | 出现“极值点”或“最大值”字样的题目
- steps: 1. 求函数的一阶导数 f'(x)。 | 2. 解方程 f'(x)=0，得到所有临界点；同时检查导数不存在的点。 | 3. 对每个临界点使用二阶导数判别法：若 f''(x)>0 为极小值，f''(x)<0 为极大值；若二阶导数为0，转用单调性法或高阶导数判别。 | 4. 计算函数在判定为极值的点处的函数值 f(x)，得到极值大小。 | 5. 若函数定义域受限，比较临界点处的函数值与

## 7. 只写到 scratch

In [10]:
if result is not None:
    (SCRATCH_DIR / 'offset_methods_rough.json').write_text(
        json.dumps(result.rough_payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    (SCRATCH_DIR / 'offset_methods_normalized.json').write_text(
        json.dumps(result.entries, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    (SCRATCH_DIR / 'offset_methods_draft.md').write_text(
        result.draft_markdown,
        encoding='utf-8',
    )
    print('saved to', SCRATCH_DIR)

saved to /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground
